# Pipeline Inputs

In [60]:
IMAGE_PATH = "data/temp/X00016469612.jpg"
DOC_CATEGORY = "receipt"
FIELDS = ["company", "date", "address", "total", "phone number"]
DEBUG = True   

In [61]:
import os, gc, re, json, time
import torch
from PIL import Image
import pytesseract
from transformers import AutoTokenizer, AutoModelForCausalLM

# Model Paths

In [62]:
pytesseract.pytesseract.tesseract_cmd = "/home/lu.dong1/.conda/envs/expo-judge/bin/tesseract"

ENGINE_A_PATH = "/projects/insightx-lab/cn_grpo/models/Llama-3.1-8B-Instruct"
ENGINE_B_PATH = "/projects/insightx-lab/cn_grpo/models/Qwen2.5-7B-Instruct"
# EVAL_MODEL_PATH = "/projects/insightx-lab/cn_grpo/models/Llama-3.1-8B-Instruct"
EVAL_MODEL_PATH = "/projects/insightx-lab/cn_grpo/models/Mistral-7B-Instruct-v0.3"

# Debug output directory (only used when DEBUG=True)
DEBUG_OUTPUT_DIR = "data/temp/debug"

# Common Utils

In [63]:
def save_text(text, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

In [64]:
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

In [65]:
def maybe_save_text(text, filename, debug, output_dir):
    if debug:
        os.makedirs(output_dir, exist_ok=True)
        save_text(text, os.path.join(output_dir, filename))

In [66]:
def maybe_save_json(obj, filename, debug, output_dir):
    if debug:
        os.makedirs(output_dir, exist_ok=True)
        save_json(obj, os.path.join(output_dir, filename))

In [67]:
def try_parse_json(text):
    for fn in [lambda t: json.loads(t),
               lambda t: json.loads(extract_json_block(t))]:
        try:
            return fn(text)
        except Exception:
            pass
    return None

In [68]:
def extract_json_block(text: str) -> str:
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return match.group(0)
    return text

In [69]:
def load_model_and_tokenizer(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    return tokenizer, model

In [70]:
def run_chat_inference(model, tokenizer, prompt: str, max_new_tokens: int = 1200) -> str:
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response

In [71]:
def release_model(model=None, tokenizer=None):
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Prompt Builders

In [72]:
def _fields_str(fields):
    """Return a bullet list string for prompt injection."""
    return "\n".join(f"- {f}" for f in fields)

In [73]:
# def _ocr_alignment_examples(fields):
#     """Dynamically generate field-aware ocr_alignment examples for Stage 2 prompt."""
#     generic = (
#         "Do NOT judge this only by whether the value appears somewhere in OCR text.\n"
#         "A value may appear in OCR text but still be weakly aligned "
#         "if it belongs to a different field context.\nExamples:"
#     )
#     examples = []
#     hints = {
#         "total": "a numeric value should be supported by total-related context, not just by appearing anywhere in OCR text.",
#         "date": "a date-like value should be supported by date-related context, not just any numeric pattern.",
#         "company": "the value should align with merchant-like or organization-like text.",
#         "address": "the value should align with address-like multi-line location text.",
#     }
#     for f in fields:
#         hint = hints.get(f, f"the value should align with {f}-related context in the OCR text.")
#         examples.append(f"- For the {f} field, {hint}")
#     return generic + "\n" + "\n".join(examples)

In [74]:
def _ocr_alignment_guideline():
    return """
OCR alignment measures whether the extracted value is grounded in OCR raw text with the correct local semantic context for the target field.

Do NOT judge this only by whether the value appears somewhere in the OCR text.

A value may appear in OCR text but still have weak OCR alignment if:
- it belongs to a different field,
- it appears under a different nearby label,
- it comes from a different section of the document,
- it is only partially matched,
- or the surrounding OCR text does not support that it corresponds to the target field.

Use the following scale:
- strong: the extracted value is clearly supported by OCR text, and the nearby context strongly indicates that it belongs to the target field.
- partial: the extracted value is partly supported by OCR text, but the context is incomplete, ambiguous, noisy, or only indirectly related to the target field.
- weak: the extracted value is unsupported, mismatched, only coincidentally present, or grounded in the wrong context.
""".strip()

### Stage-2A: Constraints Generation

In [75]:
# def build_constraints_prompt_A(raw_text, doc_category, fields):
#     return f"""
# You are a {doc_category} analysis engine.

# You are given OCR raw text extracted from a {doc_category}.

# Your goal is to infer GENERAL validation constraints for verifying extracted fields.

# Fields:
# {_fields_str(fields)}

# Instructions:
# Step 1.
# Briefly analyze what kinds of textual patterns appear in the OCR text
# (e.g., numeric patterns, address structure, date patterns).

# Step 2.
# Based on those observations, infer GENERAL validation constraints
# that could apply to many {doc_category}s.

# Important restrictions:
# - Do NOT include layout assumptions (top/bottom/middle).
# - Do NOT copy words from this {doc_category}.
# - Do NOT invent location-specific rules.
# - Constraints must describe textual properties only.

# Return ONLY JSON:

# {{
#   "inferred_constraints": {{
# {chr(10).join(f'    "{f}": [],' for f in fields[:-1])}
#     "{fields[-1]}": []
#   }}
# }}

# OCR raw text:
# {raw_text}
# """

In [76]:
# def build_constraints_prompt_A(raw_text, doc_category, fields):
#     return f"""
# You are a {doc_category} analysis engine.

# You are given OCR raw text extracted from a {doc_category}.

# Your goal is to infer GENERAL validation constraints for verifying extracted fields.

# Fields:
# {_fields_str(fields)}

# Instructions:

# Step 1.
# Briefly analyze what kinds of textual patterns appear in the OCR text
# (e.g., names, labels, numeric values, prices, dates, identifiers, short phrases,
# multi-line text blocks, punctuation patterns, or repeated formatting patterns).

# Step 2.
# Based on those observations, infer GENERAL validation constraints
# that could apply to many {doc_category}s.

# Important restrictions:
# - Do NOT copy exact words or exact values from this document as constraints.
# - Do NOT invent document-specific or location-specific rules.
# - Do NOT assume that every field appears in exactly the same form across all documents of this category.
# - Avoid relying on strict layout assumptions (e.g., fixed positions like top/bottom/left/right).
# - Layout-related patterns, if mentioned, should be treated as weak and non-deterministic signals.
# - Constraints should describe general textual or semantic properties.
# - Prefer soft, reusable plausibility rules over rigid regex-like rules.

# Return ONLY valid JSON in exactly this format:

# {{
#   "inferred_constraints": {{
# {chr(10).join(f'    "{f}": [],' for f in fields[:-1])}
#     "{fields[-1]}": []
#   }}
# }}

# OCR raw text:
# {raw_text}
# """.strip()

In [77]:
def build_constraints_prompt_A(raw_text, doc_category, fields):
    return f"""
You are a constraint inference engine for OCR-based information extraction.

Document category:
{doc_category}

Target fields:
{_fields_str(fields)}

OCR raw text:
{raw_text}

Your task:
Infer GENERAL validation constraints for each target field based on:
1. the document category
2. the OCR text patterns visible in this document
3. the likely semantic role of each field

Important:
These constraints are NOT the extracted field values.
They are reusable field-level validation hints that can later help evaluate whether a candidate value is credible.

Think at the level of:
- semantic type
- common text form
- likely context signals
- broad structural tendencies in text

Examples of useful constraint types:
- whether a field usually looks like a name, label, amount, date, identifier, short phrase, or multi-line text block
- whether a field tends to have numeric, alphabetic, mixed, or symbolic characters
- whether a field often appears with nearby cue words or local context
- whether a field is usually short, medium, or multi-line
- whether a field may contain delimiters, currency symbols, punctuation, or formatting markers

Rules:
- Be general and reusable.
- Do NOT copy exact values from the OCR text as constraints.
- Do NOT invent document-specific facts.
- Do NOT rely on exact position or page layout assumptions such as top, bottom, left, or right.
- Do NOT require strict regex-like rules unless they are truly fundamental.
- Prefer soft validation guidance over brittle hard rules.
- Keep each field's constraints concise and practical.

Return ONLY valid JSON in exactly this format:

{{
  "constraint_summary": {{
    {", ".join([f'"{field}": ["constraint 1", "constraint 2"]' for field in fields])}
  }}
}}
""".strip()

In [78]:
# def build_constraints_prompt_B(raw_text, doc_category, fields):
#     return f"""
    
# You are a {doc_category} analysis engine.

# You are given OCR raw text extracted from a {doc_category}.

# Your goal is to infer GENERAL validation constraints for verifying extracted fields.

# Fields:
# {_fields_str(fields)}

# Instructions:

# Step 1.
# Briefly analyze what kinds of textual patterns appear in the OCR text
# (e.g., numeric patterns, address structure, date patterns).

# Step 2.
# Based on those observations, infer GENERAL validation constraints
# that could apply to many {doc_category}s.

# Important restrictions:
# - Do NOT include layout assumptions (top/bottom/middle).
# - Do NOT copy words from this {doc_category}.
# - Do NOT invent location-specific rules.
# - Constraints must describe textual properties only.

# Use exactly this format:

# Step 1: Textual patterns analysis
# <Your analysis here>

# Step 2: Inferred validation constraints
# ```json
# {{
#   "inferred_constraints": {{
# {chr(10).join(f'    "{f}": [],' for f in fields[:-1])}
#     "{fields[-1]}": []
#   }}
# }}

# OCR raw text:
# {raw_text}
# """

In [79]:
def build_constraints_prompt_B(raw_text, doc_category, fields):
    return f"""
You are a constraint inference engine for OCR-based information extraction.

Document category:
{doc_category}

Target fields:
{_fields_str(fields)}

OCR raw text:
{raw_text}

Your task:
Infer GENERAL validation constraints for each target field based on:
1. the document category
2. the OCR text patterns visible in this document
3. the likely semantic role of each field

Important:
These constraints are NOT the extracted field values.
They are reusable field-level validation hints that can later help evaluate whether a candidate value is credible.

Think at the level of:
- semantic type
- common text form
- likely context signals
- broad structural tendencies in text

Examples of useful constraint types:
- whether a field usually looks like a name, label, amount, date, identifier, short phrase, or multi-line text block
- whether a field tends to have numeric, alphabetic, mixed, or symbolic characters
- whether a field often appears with nearby cue words or local context
- whether a field is usually short, medium, or multi-line
- whether a field may contain delimiters, currency symbols, punctuation, or formatting markers

Rules:
- Be general and reusable.
- Do NOT copy exact values from the OCR text as constraints.
- Do NOT invent document-specific facts.
- Do NOT rely on exact position or page layout assumptions such as top, bottom, left, or right.
- Do NOT require strict regex-like rules unless they are truly fundamental.
- Prefer soft validation guidance over brittle hard rules.
- Keep each field's constraints concise and practical.

Return ONLY valid JSON in exactly this format:

{{
  "constraint_summary": {{
    {", ".join([f'"{field}": ["constraint 1", "constraint 2"]' for field in fields])}
  }}
}}
""".strip()

In [80]:
# def build_constraints_prompt_B(raw_text, doc_category, fields):
#     return f"""
# You are a {doc_category} analysis engine.

# You are given OCR raw text extracted from a {doc_category}.

# Your goal is to infer GENERAL validation constraints for verifying extracted fields.

# Fields:
# {_fields_str(fields)}

# Instructions:

# Step 1.
# Briefly analyze what kinds of textual patterns appear in the OCR text
# (e.g., names, labels, numeric values, prices, dates, identifiers, short phrases,
# multi-line text blocks, punctuation patterns, or repeated formatting patterns).

# Step 2.
# Based on those observations, infer GENERAL validation constraints
# that could apply to many {doc_category}s.

# Important restrictions:
# - Do NOT copy exact words or exact values from this document as constraints.
# - Do NOT invent document-specific or location-specific rules.
# - Do NOT assume that every field appears in exactly the same form across all documents of this category.
# - Avoid relying on strict layout assumptions (e.g., fixed positions like top/bottom/left/right).
# - Layout-related patterns, if mentioned, should be treated as weak and non-deterministic signals.
# - Constraints should describe general textual or semantic properties.
# - Prefer soft, reusable plausibility rules over rigid regex-like rules.

# Return ONLY valid JSON in exactly this format:

# {{
#   "inferred_constraints": {{
# {chr(10).join(f'    "{f}": [],' for f in fields[:-1])}
#     "{fields[-1]}": []
#   }}
# }}

# OCR raw text:
# {raw_text}
# """.strip()

### Stage-2B: Constraint-guided Extraction

In [81]:
# def build_extraction_prompt(raw_text, constraint_trace, fields):
#     field_extraction_block = "\n".join(f'    "{f}": "",' for f in fields[:-1]) + f'\n    "{fields[-1]}": ""'
#     evidence_block = "\n".join(
#         f'    "{f}": {{\n      "evidence_text": ""\n    }}{"," if f != fields[-1] else ""}'
#         for f in fields
#     )
#     reasoning_block = "\n".join(f'    "{f}": "",' for f in fields[:-1]) + f'\n    "{fields[-1]}": ""'

#     return f"""
# You are a {", ".join(fields)} extraction engine for {DOC_CATEGORY}s.

# You are given:
# 1. OCR raw text from a {DOC_CATEGORY}
# 2. A constraint trace previously generated for this {DOC_CATEGORY}

# Your task is to extract these fields:
# {_fields_str(fields)}

# You must use the constraint trace to guide your extraction.
# For each field, output three things:
# 1. field_extraction: the final extracted value
# 2. evidence_trace: the most relevant supporting OCR text
# 3. reasoning: explain why this evidence supports the extraction, how the constraint trace helped

# Return ONLY valid JSON in exactly this format:

# {{
#   "field_extraction": {{
# {field_extraction_block}
#   }},
#   "evidence_trace": {{
# {evidence_block}
#   }},
#   "reasoning": {{
# {reasoning_block}
#   }}
# }}

# Rules:
# - Use only the OCR raw text and the constraint trace below.
# - Do not use outside knowledge.
# - If a field is uncertain, still provide the best guess, but mention the uncertainty in reasoning.
# - Keep evidence_trace short and grounded in the OCR text.
# - If the selected field value or supporting OCR evidence contains "?", preserve it exactly as-is.
# - Do NOT remove, replace, or normalize question marks.
# - Do NOT guess missing characters hidden by "?".
# - Do not wrap the output in markdown or code fences.

# Constraint trace:
# {constraint_trace}

# OCR raw text:
# {raw_text}
# """

In [82]:
def build_extraction_prompt(raw_text, constraint_trace, fields, doc_category):
    field_extraction_block = ",\n".join([f'    "{f}": ""' for f in fields])
    evidence_block = ",\n".join([f'    "{f}": ""' for f in fields])
    reasoning_block = ",\n".join([f'    "{f}": ""' for f in fields])

    return f"""
You are an information extraction engine for {doc_category} documents.

You are given:
1. OCR raw text from a {doc_category}
2. A constraint trace previously generated for this document

Your task:
Extract the following fields:
{_fields_str(fields)}

For each field, provide:
1. field_extraction: the extracted value
2. evidence_trace: the most directly relevant supporting OCR text
3. reasoning: explain why this evidence supports the extraction and how the constraint trace helped

Important guidelines:
- Extract each field independently.
- Use only the OCR raw text and the constraint trace below.
- Do NOT use outside knowledge.
- Every field listed in the schema MUST appear in the output, even if no value is found.
- If a field is partially visible or uncertain but still plausibly present, provide the best supported guess and mention the uncertainty in reasoning.
- If a field is not present in the OCR text, or no sufficiently supported candidate can be identified, you MUST return an empty string "" for that field. Do NOT omit the field.
- Do NOT hallucinate a value just to fill the schema.
- The evidence_trace for each field should be SHORT, LOCAL, and DIRECTLY GROUNDED in the OCR text.
- Do NOT use large unrelated text blocks as evidence.
- Do NOT reuse the same generic evidence for multiple fields unless it clearly supports each field.
- The extracted value should be supported by the evidence_trace for that field.
- If field_extraction for a field is "", then evidence_trace for that field must also be "", and reasoning should briefly explain that no sufficiently supported value was found.

Special rule for OCR noise:
- If the selected field value or supporting OCR evidence contains "?", preserve it exactly as-is.
- Do NOT remove, replace, or normalize question marks.
- Do NOT guess missing characters hidden by "?".

Schema completeness rules:
- You MUST return every field listed in the schema.
- Do NOT omit any field, even if the field is not present in the document.
- If a field is missing or not found, set:
  - field_extraction[field] = ""
  - evidence_trace[field] = ""
  - reasoning[field] = "No sufficiently supported value was found for this field."
- Missing fields must still appear in the JSON output.

Return ONLY valid JSON in exactly this format:

{{
  "field_extraction": {{
{field_extraction_block}
  }},
  "evidence_trace": {{
{evidence_block}
  }},
  "reasoning": {{
{reasoning_block}
  }}
}}

Constraint trace:
{constraint_trace}

OCR raw text:
{raw_text}
""".strip()

In [83]:
def build_single_extraction_prompt(raw_text, constraint_trace, field, doc_category):
    return f"""
You are an information extraction engine for {doc_category} documents.

You are given:
1. OCR raw text from a {doc_category}
2. A constraint trace previously generated for this document

Your task:
Extract ONLY this field:
- {field}

For this field, provide:
1. field_extraction: the extracted value
2. evidence_trace: the most directly relevant supporting OCR text
3. reasoning: explain why this evidence supports the extraction and how the constraint trace helped

Important guidelines:
- Use only the OCR raw text and the constraint trace below.
- Do NOT use outside knowledge.
- If the field is partially visible or uncertain but still plausibly present, provide the best supported guess and mention the uncertainty in reasoning.
- If the field is not present in the OCR text, or no sufficiently supported candidate can be identified, you MUST return an empty string "" for field_extraction.
- Do NOT hallucinate a value.
- The evidence_trace should be SHORT, LOCAL, and DIRECTLY GROUNDED in the OCR text.
- Do NOT use large unrelated text blocks as evidence.
- The extracted value should be supported by the evidence_trace.
- If field_extraction is "", then evidence_trace must also be "", and reasoning should briefly explain that no sufficiently supported value was found.

Special rule for OCR noise:
- If the selected field value or supporting OCR evidence contains "?", preserve it exactly as-is.
- Do NOT remove, replace, or normalize question marks.
- Do NOT guess missing characters hidden by "?".

Return ONLY valid JSON in exactly this format:

{{
  "field_extraction": "",
  "evidence_trace": "",
  "reasoning": ""
}}

Constraint trace:
{constraint_trace}

OCR raw text:
{raw_text}
""".strip()

### Stage-3A: Rule Synthesis

In [84]:
# def build_rule_synthesis_prompt(engineA_constraints_text, engineB_constraints_text, doc_category, fields):
#     consolidated_block = "\n".join(f'    "{f}": [],' for f in fields[:-1]) + f'\n    "{fields[-1]}": []'
    
#     return f"""
# Return ONLY valid JSON.
# Do not output explanations.

# You are a rule synthesis engine for {doc_category} field evaluation.

# Two extraction engines independently analyzed the OCR text and produced:
# - textual pattern analysis
# - inferred validation constraints

# Your task is to synthesize these into ONE shared set of consolidated checking rules
# for evaluating the credibility of extracted fields.

# Fields:
# {_fields_str(fields)}

# These rules will later be used for self-justification and cross-engine judgment.

# --------------------------------
# SYNTHESIS PRINCIPLES
# --------------------------------
# 1. Do NOT simply merge or concatenate rules from both engines.
# 2. If two rules express similar ideas, merge them into ONE generalized rule.
# 3. Prefer generalized plausibility rules rather than strict schema validation.
# 4. Avoid overly brittle rules that depend on a specific OCR instance.
# 5. Preserve useful signals such as format clues, semantic meaning, typical {doc_category} layout hints, numeric or textual patterns.
# 6. The goal is evaluation-oriented checking rules, not strict regex validation.
# 7. Each field should contain 3–5 compact checking rules.

# --------------------------------
# GOOD RULE EXAMPLES
# --------------------------------
# Good:
# "should resemble a plausible merchant name"
# "commonly follows a DD/MM/YYYY-style format"
# "may include decimal digits and optional currency notation"

# Bad:
# "pattern must be ^\\d{{2}}/\\d{{2}}/\\d{{4}}$"
# "max length must be 50"

# --------------------------------
# OUTPUT FORMAT
# --------------------------------
# Return JSON in EXACTLY this format:
# {{
#   "consolidated_rules": {{
# {consolidated_block}
#   }},
#   "synthesis_summary": {{
#     "agreement_level": "high | moderate | low",
#     "notes": ""
#   }}
# }}

# --------------------------------
# ENGINE A OUTPUT
# --------------------------------
# {engineA_constraints_text}

# --------------------------------
# ENGINE B OUTPUT
# --------------------------------
# {engineB_constraints_text}
# """

In [85]:
def build_rule_synthesis_prompt(engineA_constraints_text, engineB_constraints_text, doc_category, fields):
    consolidated_block = "\n".join(
        f'    "{f}": [],' for f in fields[:-1]
    ) + f'\n    "{fields[-1]}": []'

    return f"""
Return ONLY valid JSON.
Do not output explanations.

You are a rule synthesis engine for {doc_category} field evaluation.

Two extraction engines independently analyzed the OCR text and produced
field-level validation constraints.

Your task is to synthesize them into ONE shared set of consolidated checking rules
for evaluating the credibility of extracted fields.

Fields:
{_fields_str(fields)}

These rules will later be used for self-justification and cross-engine judgment.

--------------------------------
SYNTHESIS PRINCIPLES
--------------------------------
1. Do NOT simply merge or concatenate rules from both engines.
2. If two rules express similar ideas, merge them into ONE generalized rule.
3. Prefer generalized plausibility rules rather than strict schema validation.
4. Avoid overly brittle rules that depend on a specific OCR instance.
5. Preserve useful signals such as format clues, semantic meaning, local contextual clues in text, and numeric or textual patterns.
6. The goal is evaluation-oriented checking rules, not strict regex validation.
7. Each field should contain 3–5 compact checking rules.
8. synthesis_summary.notes should briefly summarize the main agreement or disagreement patterns between the two engines.
9. Keep synthesis_summary.notes short, concrete, and limited to one sentence.
10. Do not leave synthesis_summary.notes empty unless both engine outputs are completely uninformative.

--------------------------------
GOOD RULE EXAMPLES
--------------------------------
Good:
"should resemble a plausible value for the target field"
"may follow a common text or numeric pattern associated with the field"
"should match the expected semantic role and local text context of the field"

Bad:
"must exactly match regex ^...$"
"must appear in a fixed document position"
"must satisfy a rigid maximum length constraint"

--------------------------------
OUTPUT FORMAT
--------------------------------
Return JSON in EXACTLY this format:
{{
  "consolidated_rules": {{
{consolidated_block}
  }},
  "synthesis_summary": {{
    "agreement_level": "high | moderate | low",
    "notes": ""
  }}
}}

--------------------------------
ENGINE A OUTPUT
--------------------------------
{engineA_constraints_text}

--------------------------------
ENGINE B OUTPUT
--------------------------------
{engineB_constraints_text}
""".strip()

### Stage-3B: Self-Justification

In [86]:
# def build_stage2_prompt(field_name, rules, extracted_value, evidence_text,
#                         reasoning_text, ocr_raw_text, sanity_result, doc_category, fields):
#     return f"""
# You are an evaluation model for {doc_category} field extraction reliability.

# Your job is to evaluate whether the extracted value for one field is credible.

# Return ONLY valid JSON. No explanation.

# Field name:
# {field_name}

# Consolidated validation rules:
# {json.dumps(rules, indent=2, ensure_ascii=False)}

# Extracted value:
# {extracted_value}

# Evidence trace:
# {evidence_text}

# Engine reasoning:
# {reasoning_text}

# OCR raw text:
# {ocr_raw_text}

# Sanity check result:
# {sanity_result}

# Evaluation definitions:
# 1. rule_consistency:
# How well the extracted value matches the consolidated validation rules for this field.

# 2. engine_self_consistency:
# How well the engine's evidence trace and reasoning support its own extracted value.

# 3. ocr_alignment:
# {_ocr_alignment_guideline()}

# 4. sanity_check_result:
# Copy the provided sanity_check_result exactly.

# 5. judgment_summary:
# Write one short summary sentence explaining the overall credibility of the extracted value.
# The summary must be consistent with the other output fields.
# Do not introduce new evidence or new conclusions beyond:
# - rule_consistency
# - engine_self_consistency
# - ocr_alignment
# - sanity_check_result

# Return JSON in exactly this format:

# {{
#   "extracted_value": "",
#   "rule_consistency": "high | moderate | low",
#   "engine_self_consistency": "strong | moderate | weak",
#   "ocr_alignment": "strong | partial | weak",
#   "sanity_check_result": "pass | not_pass",
#   "judgment_summary": ""
# }}
# """

In [87]:
def build_stage2_prompt(
    field_name,
    rules,
    extracted_value,
    evidence_text,
    reasoning_text,
    ocr_raw_text,
    doc_category
):
    return f"""
You are an evaluation model for {doc_category} field extraction reliability.

Your job is to evaluate whether the extracted value for one field is credible.

Return ONLY valid JSON. No explanation.

Field name:
{field_name}

Consolidated validation rules:
{json.dumps(rules, indent=2, ensure_ascii=False)}

Extracted value:
{extracted_value}

Evidence trace:
{evidence_text}

Engine reasoning:
{reasoning_text}

OCR raw text:
{ocr_raw_text}

Evaluation definitions:
1. rule_consistency:
How well the extracted value matches the consolidated validation rules for this field.

2. engine_self_consistency:
How well the engine's evidence trace and reasoning support its own extracted value.

3. ocr_alignment:
How well the extracted value is grounded in OCR raw text with the correct local semantic context for the target field.

Use the following scale:
- strong: the value is clearly grounded in OCR text, and the surrounding OCR context strongly supports that it belongs to the target field.
- partial: the value has some OCR support, but the surrounding context is incomplete, ambiguous, noisy, or only indirectly supportive.
- weak: the value is ungrounded, mismatched, coincidental, or supported by the wrong context.

4. ocr_corruption:
Whether the extracted value itself appears to contain OCR corruption.

Judge OCR corruption mainly from the extracted value string itself.
Focus on visible recognition artifacts in the text form, not just on whether the overall meaning seems plausible.

Use the following scale:
- absent: no obvious OCR corruption is present; the value appears clean, readable, and textually well-formed.
- possible: mild or uncertain OCR corruption may be present; the value is still partly readable, but some segments, characters, or symbols are questionable.
- present: obvious OCR corruption is present; the value contains clear recognition artifacts such as unexpected "?" characters, broken segments, unusual symbol substitutions, merged fragments, or corrupted text that makes the semantic meaning unclear.

General hints:
- Unexpected "?" characters inside a word, number, or phrase are strong evidence of OCR corruption.
- Strange symbol substitutions, broken fragments, or malformed character sequences may indicate OCR corruption even if part of the value is still readable.
- A value can still be semantically plausible while being OCR-corrupted.
- Clean punctuation alone does not imply corruption.
- Normal delimiters such as commas, periods, slashes, hyphens, ampersands, or parentheses are not OCR corruption by themselves.

Examples:
- absent:
  "2023-08-15"
  "123 Main Street"
  "Total: 45.90"
- possible:
  "2O23-08-15"
  "123 Main Stree1"
  "Tota1: 45.90"
- present:
  "2O?3-0?15"
  "123 Ma?n Str?et"
  "To?al: 4?.9O"

Important:
- OCR corruption does NOT automatically mean the value is wrong.
- A value may still be partially usable even if OCR corruption is present.
- Focus on whether corruption is absent, possible, or clearly present.

5. judgment_summary:
Write one short summary sentence explaining the overall credibility of the extracted value.
The summary must be consistent with the other output fields.
Do not introduce new evidence or new conclusions beyond:
- rule_consistency
- engine_self_consistency
- ocr_alignment
- ocr_corruption

Return JSON in exactly this format:

{{
  "extracted_value": "",
  "rule_consistency": "high | moderate | low",
  "engine_self_consistency": "strong | moderate | weak",
  "ocr_alignment": "strong | partial | weak",
  "ocr_corruption": "absent | possible | present",
  "judgment_summary": ""
}}
""".strip()

### Stage-3C: Cross-Judgment

In [88]:
# def build_cross_judge_prompt_select(
#     field_name,
#     consolidated_rules_for_field,
#     engineA_field_result,
#     engineB_field_result,
#     doc_category
# ):
#     return f"""
# Return ONLY valid JSON.
# Do not output explanations.

# You are a cross-engine selection model for {doc_category} field extraction reliability.

# Your task is to compare the self-justification results from engineA and engineB
# for one field, and decide which engine provides the more credible extraction.

# Field name:
# {field_name}

# Consolidated checking rules:
# {json.dumps(consolidated_rules_for_field, indent=2, ensure_ascii=False)}

# EngineA result:
# {json.dumps(engineA_field_result, indent=2, ensure_ascii=False)}

# EngineB result:
# {json.dumps(engineB_field_result, indent=2, ensure_ascii=False)}

# Instructions:
# 1. Compare the two engines based on:
#    - extracted_value
#    - rule_consistency
#    - engine_self_consistency
#    - ocr_alignment
#    - ocr_corruption
# 2. Select exactly one:
#    - "engineA"
#    - "engineB"
#    - "none"
# 3. Use "none" only when neither engine provides a usable candidate.
# 4. selection_reason must be one short sentence and must not be empty.

# Return JSON in exactly this format:
# {{
#   "selected_engine": "",
#   "selection_reason": ""
# }}
# """.strip()

In [89]:
# def build_cross_judge_prompt_state(
#     field_name,
#     merged_result,
#     doc_category
# ):
#     return f"""
# Return ONLY valid JSON.
# Do not output explanations.

# You are a field-state assessment model for {doc_category} field extraction reliability.

# Your task is to assess the final state of one already-selected field result.

# Field name:
# {field_name}

# Merged field result:
# {json.dumps(merged_result, indent=2, ensure_ascii=False)}

# Field confidence definition:
# field_confidence should reflect the overall credibility of the recommended value,
# based on:
# - final_rule_consistency
# - final_engine_self_consistency
# - final_ocr_alignment
# - final_ocr_corruption

# Use the following scale:
# - very_high: all major support signals are very strong, and OCR corruption is absent.
# - high: support signals are strong overall, and OCR corruption is absent.
# - medium: the value is still plausible, but one or more support signals is not strong,
#   or OCR corruption is possible.
# - low: the value is weakly supported, OCR corruption is present, or no usable value can be recommended.

# Field state definitions:
# - pass:
#   use this only when OCR corruption is absent and the three main support signals
#   (rule consistency, engine self-consistency, OCR alignment) are all strong.
# - review_needed:
#   use this when a usable candidate still exists, but OCR corruption is possible or present,
#   or one or more of the three support signals is not strong.
# - fail:
#   use this when no usable candidate value is available,
#   or when the three support signals are collectively too weak to justify a credible recommendation.

# Important clarification:
# - OCR corruption means recognition artifacts such as unexpected "?" characters,
#   unusual symbol substitutions, broken segments, or corrupted text that makes the semantic meaning unclear.
# - OCR corruption does NOT automatically mean fail.
# - A non-empty but OCR-corrupted value should usually be labeled review_needed, not fail.
# - Example: "NO.5? $5,57 & 59, JALAN SAGU 18" is OCR-corrupted text and should usually be treated as review_needed if it still provides a partially usable candidate.

# State reason:
# - state_reason explains why the final field_state is "review_needed" or "fail".
# - If field_state = "pass", state_reason must be an empty string.
# - If field_state = "review_needed" or "fail", state_reason must be a short, specific reason.

# Decision rules:
# - If selected_engine = "none" or recommended_value is empty, use field_state = "fail".
# - If the recommended value is non-empty but OCR corruption is possible or present, use field_state = "review_needed".
# - If OCR corruption is absent, and all three main support signals are strong, use field_state = "pass".
# - If OCR corruption is absent, but one or more of the three support signals is not strong, use field_state = "review_needed".
# - Use fail only when no usable recommendation exists, or the support signals are collectively too weak.

# Consistency constraints:
# - If field_state = "pass", field_confidence should be "high" or "very_high".
# - If field_state = "review_needed", field_confidence should be "medium" or "low".
# - If field_state = "fail", field_confidence must be "low".

# Return JSON in exactly this format:
# {{
#   "field_confidence": "very_high | high | medium | low",
#   "field_state": "pass | review_needed | fail",
#   "state_reason": ""
# }}
# """.strip()

In [110]:
def build_cross_judge_prompt(
    field_name,
    consolidated_rules_for_field,
    engineA_field_result,
    engineB_field_result,
    doc_category
):
    return f"""
Return ONLY valid JSON.
Do not output explanations.

You are a cross-engine judgment model for {doc_category} field extraction reliability.

Your task is to compare the self-justification results from engineA and engineB
for one field, and decide which extracted value is more credible.

Field name:
{field_name}

Consolidated checking rules:
{json.dumps(consolidated_rules_for_field, indent=2, ensure_ascii=False)}

EngineA result:
{json.dumps(engineA_field_result, indent=2, ensure_ascii=False)}

EngineB result:
{json.dumps(engineB_field_result, indent=2, ensure_ascii=False)}

Judgment instructions:
1. Compare the two candidate extractions and their stage-3B signals.
2. Select the more credible extracted value.
3. selected_engine must be exactly one of:
   - "engineA"
   - "engineB"
   - "none"
4. Use selected_engine = "none" only if neither engine provides a usable recommendation.
5. If selected_engine = "none", recommended_value must be an empty string.

Field confidence definition:
field_confidence should reflect the overall credibility of the recommended value,
based on:
- final_rule_consistency
- final_engine_self_consistency
- final_ocr_alignment
- final_ocr_corruption

Use the following scale:
- very_high: all major support signals are very strong, and OCR corruption is absent.
- high: support signals are strong overall, and OCR corruption is absent.
- medium: the value is still plausible, but one or more support signals is not strong,
  or OCR corruption is possible.
- low: the value is weakly supported, OCR corruption is present, or no usable value can be recommended.

Field state definitions:
- pass:
  use this only when OCR corruption is absent and the three main support signals
  (rule consistency, engine self-consistency, OCR alignment) are all strong.
- review_needed:
  use this when a usable candidate still exists, but OCR corruption is possible or present,
  or one or more of the three support signals is not strong.
- fail:
  use this when no usable candidate value is available,
  or when the three support signals are collectively too weak to justify a credible recommendation.

Important clarification:
- OCR corruption means recognition artifacts such as unexpected "?" characters,
  unusual symbol substitutions, broken segments, or corrupted text that makes the semantic meaning unclear.
- OCR corruption does NOT automatically mean fail.
- A non-empty but OCR-corrupted value should usually be labeled review_needed, not fail.
- Example: "NO.5? $5,57 & 59, JALAN SAGU 18" is OCR-corrupted text and should usually be treated as review_needed if it still provides a partially usable candidate.

State reason:
- state_reason explains why the final field_state is "review_needed" or "fail".
- If field_state = "pass", state_reason must be an empty string.
- If field_state = "review_needed" or "fail", state_reason must be a short, specific reason.

Decision rules:
- If selected_engine = "none" or recommended_value is empty, use field_state = "fail".
- If the recommended value is non-empty but OCR corruption is possible or present, use field_state = "review_needed".
- If OCR corruption is absent, and all three main support signals are strong, use field_state = "pass".
- If OCR corruption is absent, but one or more of the three support signals is not strong, use field_state = "review_needed".
- Use fail only when no usable recommendation exists, or the support signals are collectively too weak.

Consistency constraints:
- If field_state = "pass", field_confidence should be "high" or "very_high".
- If field_state = "review_needed", field_confidence should be "medium" or "low".
- If field_state = "fail", field_confidence must be "low".
- Do NOT output multiple engines.
- Do NOT output "engineA | engineB".

Return JSON in exactly this format:
{{
  "recommended_value": "",
  "selected_engine": "",
  "selection_reason": "",
  "final_rule_consistency": "high | moderate | low",
  "final_engine_self_consistency": "strong | moderate | weak",
  "final_ocr_alignment": "strong | partial | weak",
  "final_ocr_corruption": "absent | possible | present",
  "field_confidence": "very_high | high | medium | low",
  "field_state": "pass | review_needed | fail",
  "state_reason": ""
}}
""".strip()

In [91]:

# def build_cross_judge_prompt(field_name, consolidated_rules_for_field,
#                               engineA_field_result, engineB_field_result, doc_category):
#     return f"""
# Return ONLY valid JSON.
# Do not output explanations.

# You are a cross-engine judgment model for {doc_category} field extraction reliability.
# Your task is to compare the stage-2 self-justification results from engineA and engineB
# for one field, and decide which extracted value is more credible.
# Field name:
# {field_name}

# Consolidated checking rules:
# {json.dumps(consolidated_rules_for_field, indent=2, ensure_ascii=False)}

# EngineA result:
# {json.dumps(engineA_field_result, indent=2, ensure_ascii=False)}

# EngineB result:
# {json.dumps(engineB_field_result, indent=2, ensure_ascii=False)}

# Judgment instructions:
# 1. Compare the two extracted values and their stage-2 signals.
# 2. Select the more credible value.
# 3. If both are unreliable, selected_engine can be "none".
# 4. If selected_engine is "none", set recommended_value to an empty string.
# 5. final_rule_consistency should reflect the final recommended value.
# 6. final_engine_self_consistency should reflect how trustworthy the selected engine's self-justification is.
# 7. final_ocr_alignment should reflect the selected value's OCR support.
# 8. field_confidence should be one of: very_high / high / medium / low
# 9. field_state should be one of: pass / review_needed / fail

# Guidance:
# - Prefer values with stronger rule consistency, stronger self-consistency, stronger OCR alignment, and passing sanity check.
# - If a value has sanity_check_result = not_pass, treat it as suspicious.
# - If both values are weak or unsupported, use selected_engine = "none".

# Consistency rules:
# - field_confidence and field_state must be mutually consistent.
# - If field_state = "pass", field_confidence should must be "high" or "very_high".
# - If field_state = "review_needed", field_confidence should must be "medium" or "low".
# - If field_state = "fail", field_confidence must be "low".
# - If selected_engine = "none", field_confidence must be "low" and field_state should must be "fail" or "review_needed".

# Return JSON in exactly this format:
# {{
#   "recommended_value": "",
#   "selected_engine": "engineA | engineB | none",
#   "selection_reason": "",
#   "final_rule_consistency": "high | moderate | low",
#   "final_engine_self_consistency": "strong | moderate | weak",
#   "final_ocr_alignment": "strong | partial | weak",
#   "field_confidence": "very_high | high | medium | low",
#   "field_state": "pass | review_needed | fail"
# }}
# """

In [92]:
# def build_cross_judge_prompt(
#     field_name,
#     consolidated_rules_for_field,
#     engineA_field_result,
#     engineB_field_result,
#     doc_category
# ):
#     return f"""
# Return ONLY valid JSON.
# Do not output explanations.

# You are a cross-engine judgment model for {doc_category} field extraction reliability.

# Your task is to compare the stage-2 self-justification results from engineA and engineB
# for one field, and decide which candidate extraction is more credible overall.

# Field name:
# {field_name}

# Consolidated checking rules:
# {json.dumps(consolidated_rules_for_field, indent=2, ensure_ascii=False)}

# EngineA result:
# {json.dumps(engineA_field_result, indent=2, ensure_ascii=False)}

# EngineB result:
# {json.dumps(engineB_field_result, indent=2, ensure_ascii=False)}

# Judgment instructions:
# 1. Compare the two candidate extractions and their stage-2 signals.
# 2. Select the more credible value.
# 3. Use selected_engine = "none" only when neither engine provides a usable candidate value.
# 4. If selected_engine is "none", recommended_value must be an empty string.
# 5. final_rule_consistency should reflect the final recommended value.
# 6. final_engine_self_consistency should reflect how trustworthy the selected engine's self-justification is.
# 7. final_ocr_alignment should reflect the selected value's OCR support.
# 8. field_confidence must be one of: very_high, high, medium, low
# 9. field_state must be one of: pass, review_needed, fail
# 10. state_reason explains why the final field_state is review_needed or fail.
# 11. If field_state = "pass", state_reason should be an empty string.
# 12. If field_state = "review_needed" or "fail", state_reason must be a short, specific reason.

# Field state definitions:
# - pass: the recommended value is sufficiently credible and does not need manual review.
# - review_needed: there is a partially credible candidate, but uncertainty, ambiguity, OCR corruption, or unresolved conflict still requires manual review.
# - fail: neither engine provides a sufficiently credible candidate, so no usable recommendation can be made.

# Important:
# - A sanity_check_result of not_pass does NOT automatically mean fail.
# - If a candidate still has meaningful rule support, self-consistency, or OCR alignment but remains uncertain, prefer review_needed over fail.
# - Use fail only when both candidates are too weak, unsupported, or unusable to recommend.

# Decision order:
# 1. First determine whether there is a usable recommended_value.
# 2. Then determine field_state.
# 3. Then assign field_confidence so that it is fully consistent with field_state.
# 4. Then choose selected_engine.
# 5. Then write selection_reason and state_reason.

# Consistency constraints:
# - field_state and field_confidence must be strictly consistent.
# - If field_state = "pass", field_confidence must be "high" or "very_high".
# - If field_state = "review_needed", field_confidence must be "medium" or "low". Do NOT output "high" or "very_high".
# - If field_state = "fail", field_confidence must be "low".
# - If selected_engine = "none", field_confidence must be "low".

# selected_engine constraints:
# - selected_engine must be exactly one of these three strings:
#   "engineA"
#   "engineB"
#   "none"
# - Do NOT output multiple engines.
# - Do NOT output "engineA | engineB".

# Guidance:
# - Prefer values with stronger rule consistency, stronger self-consistency, stronger OCR alignment, and a more favorable sanity_check_result.
# - Treat sanity_check_result = not_pass as a negative signal, but not as an automatic failure.
# - If one candidate is still partially supported but uncertain, you may still select that engine and set field_state = "review_needed".

# Return JSON in exactly this format:
# {{
#   "recommended_value": "",
#   "selected_engine": "",
#   "selection_reason": "",
#   "final_rule_consistency": "high | moderate | low",
#   "final_engine_self_consistency": "strong | moderate | weak",
#   "final_ocr_alignment": "strong | partial | weak",
#   "field_confidence": "very_high | high | medium | low",
#   "field_state": "pass | review_needed | fail",
#   "state_reason": ""
# }}
# """.strip()

In [93]:
# def build_cross_judge_prompt(
#     field_name,
#     consolidated_rules_for_field,
#     engineA_field_result,
#     engineB_field_result,
#     doc_category
# ):
#     return f"""
# Return ONLY valid JSON.
# Do not output explanations.

# You are a cross-engine judgment model for {doc_category} field extraction reliability.

# Your task is to compare the stage-2 self-justification results from engineA and engineB
# for one field, and decide which candidate extraction is more credible overall.

# Field name:
# {field_name}

# Consolidated checking rules:
# {json.dumps(consolidated_rules_for_field, indent=2, ensure_ascii=False)}

# EngineA result:
# {json.dumps(engineA_field_result, indent=2, ensure_ascii=False)}

# EngineB result:
# {json.dumps(engineB_field_result, indent=2, ensure_ascii=False)}

# Judgment instructions:
# 1. Compare the two candidate extractions and their stage-2 signals.
# 2. Select the more credible candidate extraction.
# 3. selected_engine must be exactly one of:
#    - "engineA"
#    - "engineB"
#    - "none"
# 4. Use selected_engine = "none" only if neither engine provides a usable recommendation.
# 5. If selected_engine = "none", recommended_value must be an empty string.
# 6. final_rule_consistency should reflect the final recommended value.
# 7. final_engine_self_consistency should reflect how trustworthy the selected engine's self-justification is.
# 8. final_ocr_alignment should reflect the selected value's OCR support.
# 9. field_confidence must be one of:
#    - "very_high"
#    - "high"
#    - "medium"
#    - "low"
# 10. field_state must be one of:
#    - "pass"
#    - "review_needed"
#    - "fail"
# 11. state_reason explains why the final field_state is review_needed or fail.
# 12. If field_state = "pass", state_reason should be an empty string.
# 13. If field_state = "review_needed" or "fail", state_reason must be a short, specific reason.

# Field state definitions:
# - pass:
#   the recommended value is sufficiently credible and does not need manual review.
# - review_needed:
#   a non-empty candidate exists, but OCR corruption, unclear symbols, ambiguity,
#   partial semantic uncertainty, or unresolved conflict still requires human review.
# - fail:
#   no usable candidate value is available, or the recommended value is effectively missing.

# Definition of OCR corruption:
# OCR corruption means the value is non-empty but contains obvious recognition artifacts
# that make the text partially unclear or unreliable, such as:
# - unexpected "?" characters inside a word or number
# - unusual symbol substitutions
# - broken or partially corrupted segments
# - punctuation or characters that make the semantic meaning uncertain

# Sanity-check guidance:
# - sanity_check_result is only a strong signal about whether a candidate may be missing or unusable.
# - A sanity_check_result of "not_pass" should be treated as a strong negative signal.
# - If the recommended candidate is empty or effectively missing, prefer field_state = "fail".
# - However, a non-empty candidate with OCR corruption, unclear symbols, or partial uncertainty
#   should usually be labeled "review_needed", not automatically "fail".
# - Do NOT treat every imperfect OCR artifact as automatic failure.

# Decision guidance:
# - Prefer values with stronger rule consistency, stronger self-consistency,
#   stronger OCR alignment, and better overall usability.
# - If one candidate is imperfect but still more usable and better supported than the other,
#   you may still select that engine and set field_state = "review_needed".
# - Use selected_engine = "none" only when neither engine provides a reasonable candidate for recommendation.

# Consistency constraints:
# - field_state and field_confidence must be mutually consistent.
# - If field_state = "pass", field_confidence should be "high" or "very_high".
# - If field_state = "review_needed", field_confidence should be "medium" or "low".
# - If field_state = "fail", field_confidence must be "low".
# - If selected_engine = "none", field_confidence must be "low".
# - Do NOT output multiple engines.
# - Do NOT output "engineA | engineB".

# Return JSON in exactly this format:
# {{
#   "recommended_value": "",
#   "selected_engine": "",
#   "selection_reason": "",
#   "final_rule_consistency": "high | moderate | low",
#   "final_engine_self_consistency": "strong | moderate | weak",
#   "final_ocr_alignment": "strong | partial | weak",
#   "field_confidence": "very_high | high | medium | low",
#   "field_state": "pass | review_needed | fail",
#   "state_reason": ""
# }}
# """.strip()

# Pipeline Stages

In [94]:
# def run_sanity_check(value):
#     value = str(value).strip() if value is not None else ""
#     if not value or "?" in value:
#         return "not_pass"
#     return "pass"

In [95]:
def run_sanity_check(value):
    value = str(value).strip() if value is not None else ""
    if not value:
        return "not_pass"
    return "pass"

In [96]:
def normalize_extraction_output(extraction, fields):
    extraction = extraction or {}

    field_extraction = extraction.get("field_extraction", {})
    evidence_trace = extraction.get("evidence_trace", {})
    reasoning = extraction.get("reasoning", {})

    for field in fields:
        if field not in field_extraction:
            field_extraction[field] = ""
        if field not in evidence_trace:
            evidence_trace[field] = ""
        if field not in reasoning:
            reasoning[field] = "No sufficiently supported value was found for this field."

    extraction["field_extraction"] = field_extraction
    extraction["evidence_trace"] = evidence_trace
    extraction["reasoning"] = reasoning
    return extraction

In [97]:
def phase_ocr(image_path):
    image = Image.open(image_path)
    return pytesseract.image_to_string(image)

In [98]:
def phase_constraints(raw_text, doc_category, fields, debug, output_dir, base_name):
    results = {}
    
    for engine_name, engine_path, prompt_fn in [
        ("engineA", ENGINE_A_PATH, build_constraints_prompt_A),
        ("engineB", ENGINE_B_PATH, build_constraints_prompt_B),
    ]:
        prompt = prompt_fn(raw_text, doc_category, fields)
        tokenizer, model = load_model_and_tokenizer(engine_path)
        
        response = run_chat_inference(
            model,
            tokenizer,
            prompt,
            max_new_tokens=800
        )
        
        release_model(model, tokenizer)
        maybe_save_text(response, f"{base_name}_{engine_name}_constraints.txt", debug, output_dir)
        
        results[engine_name] = response
        print(response)
        
    return results

In [99]:
def phase_extraction(raw_text, constraints, fields, debug, output_dir, base_name):
    results = {}
    
    engine_paths = {
        "engineA": ENGINE_A_PATH,
        "engineB": ENGINE_B_PATH
    }
    
    for engine_name, constraint_trace in constraints.items():
        prompt = build_extraction_prompt(raw_text, constraint_trace, fields, DOC_CATEGORY)
        
        tokenizer, model = load_model_and_tokenizer(engine_paths[engine_name])
        response = run_chat_inference(
            model,
            tokenizer,
            prompt,
            max_new_tokens=1600
        )
        
        release_model(model, tokenizer)
        
        extraction = json.loads(extract_json_block(response))
        # extraction = normalize_extraction_output(extraction, fields)

        maybe_save_json(extraction, f"{base_name}_{engine_name}_extraction.json", debug, output_dir)
        
        results[engine_name] = extraction
        print(extraction)
        
    return results

In [108]:
def phase_extraction(raw_text, constraints, fields, debug, output_dir, base_name):
    results = {}

    engine_paths = {
        "engineA": ENGINE_A_PATH,
        "engineB": ENGINE_B_PATH
    }

    for engine_name, constraint_trace in constraints.items():
        tokenizer, model = load_model_and_tokenizer(engine_paths[engine_name])

        extraction = {
            "field_extraction": {},
            "evidence_trace": {},
            "reasoning": {}
        }

        for field in fields:
            prompt = build_single_extraction_prompt(
                raw_text,
                constraint_trace,
                field,
                DOC_CATEGORY
            )

            response = run_chat_inference(
                model,
                tokenizer,
                prompt,
                max_new_tokens=500
            )

            try:
                parsed = json.loads(extract_json_block(response))

                extraction["field_extraction"][field] = parsed.get("field_extraction", "")
                extraction["evidence_trace"][field] = parsed.get("evidence_trace", "")
                extraction["reasoning"][field] = parsed.get(
                    "reasoning",
                    "No sufficiently supported value was found for this field."
                )

            except Exception as e:
                print(f"[{engine_name}] failed to parse field '{field}': {e}")
                print(response)

                extraction["field_extraction"][field] = ""
                extraction["evidence_trace"][field] = ""
                extraction["reasoning"][field] = "No sufficiently supported value was found for this field."

        release_model(model, tokenizer)

        maybe_save_json(
            extraction,
            f"{base_name}_{engine_name}_extraction.json",
            debug,
            output_dir
        )

        results[engine_name] = extraction
        print(f"\n===== {engine_name} extraction =====")
        print(json.dumps(extraction, indent=2, ensure_ascii=False))

    return results

In [101]:
def phase_rule_synthesis(constraints, doc_category, fields, debug, output_dir, base_name):
    prompt = build_rule_synthesis_prompt(
        constraints["engineA"],
        constraints["engineB"],
        doc_category, fields
    )
    
    tokenizer, model = load_model_and_tokenizer(EVAL_MODEL_PATH)
    response = run_chat_inference(
        model,
        tokenizer,
        prompt,
        max_new_tokens=800
    )

    print(response)
    
    release_model(model, tokenizer)
    consolidated = json.loads(extract_json_block(response))
    
    maybe_save_json(consolidated, f"{base_name}_eval_constraints.json", debug, output_dir)
    
    return consolidated

In [102]:
# def phase_self_justification(extractions, consolidated_rules, raw_text, doc_category, fields, debug, output_dir, base_name):
#     results = {}
    
#     for engine_name, extraction in extractions.items():
#         engine_result = {
#             "engine": engine_name, 
#             "field_results": {}
#         }
        
#         for field in fields:
#             extracted_value = extraction["field_extraction"][field]
#             evidence_text = extraction["evidence_trace"][field]["evidence_text"]
#             reasoning_text = extraction["reasoning"][field]
#             rules = consolidated_rules["consolidated_rules"][field]
#             sanity_result = run_sanity_check(extracted_value)

#             prompt = build_stage2_prompt(
#                 field, rules, extracted_value, evidence_text,
#                 reasoning_text, raw_text, sanity_result, doc_category, fields
#             )
            
#             tokenizer, model = load_model_and_tokenizer(EVAL_MODEL_PATH)
#             response = run_chat_inference(model, tokenizer, prompt, max_new_tokens=600)
            
#             release_model(model, tokenizer)
#             engine_result["field_results"][field] = json.loads(extract_json_block(response))

#         maybe_save_json(engine_result, f"{base_name}_{engine_name}_self_judge.json", debug, output_dir)
        
#         results[engine_name] = engine_result
        
#     return results

In [103]:
def phase_self_justification(extractions, consolidated_rules, raw_text, doc_category, fields, debug, output_dir, base_name):
    results = {}
    
    for engine_name, extraction in extractions.items():
        engine_result = {
            "engine": engine_name,
            "field_results": {}
        }
        
        for field in fields:
            extracted_value = extraction["field_extraction"][field]
            evidence_text = extraction["evidence_trace"][field]
            reasoning_text = extraction["reasoning"][field]
            rules = consolidated_rules["consolidated_rules"][field]
            sanity_result = run_sanity_check(extracted_value)

            prompt = build_stage2_prompt(
                field_name=field,
                rules=rules,
                extracted_value=extracted_value,
                evidence_text=evidence_text,
                reasoning_text=reasoning_text,
                ocr_raw_text=raw_text,
                doc_category=doc_category,
            )
            
            tokenizer, model = load_model_and_tokenizer(EVAL_MODEL_PATH)
            response = run_chat_inference(model, tokenizer, prompt, max_new_tokens=600)
            
            release_model(model, tokenizer)
            engine_result["field_results"][field] = json.loads(extract_json_block(response))

        maybe_save_json(engine_result, f"{base_name}_{engine_name}_self_judge.json", debug, output_dir)
        results[engine_name] = engine_result
        
    return results

In [104]:
def phase_cross_judgment(self_justifications, consolidated_rules, doc_category, fields, debug, output_dir, base_name):
    stage3 = {
        "field_results": {}
    }
    
    for field in fields:
        engineA_result = self_justifications["engineA"]["field_results"][field]
        engineB_result = self_justifications["engineB"]["field_results"][field]
        rules = consolidated_rules["consolidated_rules"][field]

        prompt = build_cross_judge_prompt(
            field, rules, engineA_result, engineB_result, doc_category
        )
        tokenizer, model = load_model_and_tokenizer(EVAL_MODEL_PATH)
        response = run_chat_inference(model, tokenizer, prompt, max_new_tokens=600)
        release_model(model, tokenizer)
        stage3["field_results"][field] = json.loads(extract_json_block(response))

    maybe_save_json(stage3, f"{base_name}_cross_judge.json", debug, output_dir)
    
    return stage3

In [105]:
# def phase_cross_judgment(self_justifications, consolidated_rules, doc_category, fields, debug, output_dir, base_name):
#     stage3 = {
#         "field_results": {}
#     }
    
#     for field in fields:
#         engineA_result = self_justifications["engineA"]["field_results"][field]
#         engineB_result = self_justifications["engineB"]["field_results"][field]
#         rules = consolidated_rules["consolidated_rules"][field]

#         # ---------- Step 1: select engine ----------
#         prompt_select = build_cross_judge_prompt_select(
#             field_name=field,
#             consolidated_rules_for_field=rules,
#             engineA_field_result=engineA_result,
#             engineB_field_result=engineB_result,
#             doc_category=doc_category
#         )

#         tokenizer, model = load_model_and_tokenizer(EVAL_MODEL_PATH)
#         response_select = run_chat_inference(model, tokenizer, prompt_select, max_new_tokens=300)
#         release_model(model, tokenizer)

#         select_result = json.loads(extract_json_block(response_select))
#         selected_engine = select_result.get("selected_engine", "none")
#         selection_reason = select_result.get("selection_reason", "")

#         # ---------- Step 2: python merge ----------
#         if selected_engine == "engineA":
#             merged_result = {
#                 "recommended_value": engineA_result.get("extracted_value", ""),
#                 "selected_engine": "engineA",
#                 "selection_reason": selection_reason,
#                 "final_rule_consistency": engineA_result.get("rule_consistency", "low"),
#                 "final_engine_self_consistency": engineA_result.get("engine_self_consistency", "weak"),
#                 "final_ocr_alignment": engineA_result.get("ocr_alignment", "weak"),
#                 "final_ocr_corruption": engineA_result.get("ocr_corruption", "possible")
#             }
#         elif selected_engine == "engineB":
#             merged_result = {
#                 "recommended_value": engineB_result.get("extracted_value", ""),
#                 "selected_engine": "engineB",
#                 "selection_reason": selection_reason,
#                 "final_rule_consistency": engineB_result.get("rule_consistency", "low"),
#                 "final_engine_self_consistency": engineB_result.get("engine_self_consistency", "weak"),
#                 "final_ocr_alignment": engineB_result.get("ocr_alignment", "weak"),
#                 "final_ocr_corruption": engineB_result.get("ocr_corruption", "possible")
#             }
#         else:
#             merged_result = {
#                 "recommended_value": "",
#                 "selected_engine": "none",
#                 "selection_reason": selection_reason,
#                 "final_rule_consistency": "low",
#                 "final_engine_self_consistency": "weak",
#                 "final_ocr_alignment": "weak",
#                 "final_ocr_corruption": "present"
#             }

#         # ---------- Step 3: assess state ----------
#         prompt_state = build_cross_judge_prompt_state(
#             field_name=field,
#             merged_result=merged_result,
#             doc_category=doc_category
#         )

#         tokenizer, model = load_model_and_tokenizer(EVAL_MODEL_PATH)
#         response_state = run_chat_inference(model, tokenizer, prompt_state, max_new_tokens=300)
#         release_model(model, tokenizer)

#         state_result = json.loads(extract_json_block(response_state))

#         final_result = {
#             **merged_result,
#             **state_result
#         }

#         stage3["field_results"][field] = final_result

#     maybe_save_json(stage3, f"{base_name}_cross_judge.json", debug, output_dir)
    
#     return stage3

# Main Pipeline

In [106]:
import datetime

def run_pipeline(image_path, doc_category, fields, debug=False):
    
    pipeline_start = time.time()
    base_name  = os.path.splitext(os.path.basename(image_path))[0]
    output_dir = DEBUG_OUTPUT_DIR

    print("[1/5] OCR extraction...")
    raw_text = phase_ocr(image_path)
    maybe_save_text(raw_text, f"{base_name}_raw_text.txt", debug, output_dir)
    print(raw_text)

    print("[2/5] Constraint generation (Engine A & B)...")
    constraints = phase_constraints(raw_text, doc_category, fields, debug, output_dir, base_name)

    print("[3/5] Constraint-guided extraction (Engine A & B)...")
    extractions = phase_extraction(raw_text, constraints, fields, debug, output_dir, base_name)

    print("[4/5] Rule synthesis + self-justification...")
    consolidated_rules  = phase_rule_synthesis(constraints, doc_category, fields, debug, output_dir, base_name)
    self_justifications = phase_self_justification(
        extractions, consolidated_rules, raw_text, doc_category, fields, debug, output_dir, base_name
    )

    print("[5/5] Cross-engine judgment...")
    cross_result = phase_cross_judgment(
        self_justifications, consolidated_rules, doc_category, fields, debug, output_dir, base_name
    )

    final_result = {
        "metadata": {
            "image_path": image_path,
            "doc_category": doc_category,
            "fields": fields,
            "debug": debug,
            "timestamp": datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
            "elapsed_seconds": round(time.time() - pipeline_start, 1),
        },
        "field_results": cross_result["field_results"],
    }

    maybe_save_json(final_result, f"{base_name}_final_result.json", debug, output_dir)

    print("\n=== FINAL RESULT ===")
    print(json.dumps(final_result, indent=2, ensure_ascii=False))
    
    return final_result

In [112]:
result = run_pipeline(
    image_path = IMAGE_PATH,
    doc_category = DOC_CATEGORY,
    fields = FIELDS,
    debug = DEBUG
)

[1/5] OCR extraction...
tan woon yann

BOOK TA -K (TAMAN DAYA) SDN BHD
TBO7-W
NO.5? $5,57 & 59, JALAN SAGU 18,
TAMAN DAYA
81100 JOHOR BAHRU.
JOHOR.

LAM MCAT

Document Ho : TDO1167104

Date 25/12/2018 8:13:39 PM
Cashier MANIS
Member
CASH BILL
CODE/DESC PRICE Dise AMOUNT
Quy RM RM
9556939040118 AF MODELLING CLAY KIDDY FiSHt
1PC + 9.00) 0,00 9.00
Total : 9,00
Rour ding Adjustment 0.00
Round. :d Total (RM): 9.00
Cash a 40.00.
CHANGE 00

GOODS SOLD ARE NOT RETURNAP
EXCHANGEABLE

THANK YOU
PLEASE COME AGAIN t

[2/5] Constraint generation (Engine A & B)...


Loading weights: 100%|██████████| 291/291 [00:04<00:00, 66.81it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


```json
{
  "constraint_summary": {
    "company": [
      "Typically appears as a name with a mix of alphabetic characters, possibly with a suffix like 'SDN BHD'",
      "Often includes words like 'TA', 'SDN', or 'BHD' indicating a company name",
      "May be a multi-line text block"
    ],
    "date": [
      "Usually in the format 'DD/MM/YYYY' or 'DD/MM/YYYY HH:MM:SS'",
      "May include time in 24-hour format",
      "Often appears near the top of the document"
    ],
    "address": [
      "Typically includes a street number, street name, and postal code",
      "May include words like 'JALAN', 'TAMAN', or 'JOHOR' indicating a location",
      "Often appears on multiple lines"
    ],
    "total": [
      "Usually a numeric value with a currency symbol like 'RM'",
      "May include a decimal point and two digits after the decimal point",
      "Often appears near the bottom of the document"
    ],
    "phone number": [
      "Not present in this document, but typically a sequenc

Loading weights: 100%|██████████| 339/339 [00:03<00:00, 85.43it/s, Materializing param=model.norm.weight]                               


```json
{
  "constraint_summary": {
    "company": ["usually contains a business name", "often includes 'SDN BHD'", "appears near the top of the document", "may include alphanumeric characters and special characters like hyphens and parentheses"], 
    "date": ["formatted as day/month/year", "often includes time", "appears near the top or in the header section", "may include colons and slashes"], 
    "address": ["multi-line text block", "includes street numbers, names, and city/town", "appears below the company name", "contains alphanumeric characters and special characters like commas and periods"], 
    "total": ["numeric value with currency symbol 'RM'", "ends with a total amount", "appears near the bottom of the document", "may include decimal points"], 
    "phone number": ["numeric value", "often includes country code or area code", "may appear in the address or contact section", "can be formatted with spaces or hyphens"]
  }
}
```
[3/5] Constraint-guided extraction (Engine A & 

...

{
  "consolidated_rules": {
    "company": [
      "Typically appears as a name with a mix of alphabetic characters, possibly with a suffix like 'SDN BHD' or 'Sdn Bhd'",
      "Often includes words like 'TA', 'SDN', or 'BHD' indicating a company name",
      "May be a multi-line text block and may include alphanumeric characters and special characters like hyphens and parentheses"
    ],
    "date": [
      "Usually in the format 'DD/MM/YYYY' or 'DD/MM/YYYY HH:MM:SS', may include time in 24-hour format",
      "Often appears near the top of the document or in the header section, and may include colons and slashes"
    ],
    "address": [
      "Typically includes a street number, street name, and postal code",
      "May include words like 'JALAN', 'TAMAN', or 'JOHOR' indicating a location",
      "Often appears below the company name, on multiple lines, and contains alphanumeric characters and special characters like commas and periods"
    ],
    "total": [
      "Usually a numeric v

Loading weights: 100%|██████████| 291/291 [00:03<00:00, 74.70it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 73.29it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 75.18it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 77.23it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 74.14it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for o

[5/5] Cross-engine judgment...


Loading weights: 100%|██████████| 291/291 [00:03<00:00, 75.83it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 74.53it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 73.57it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 76.60it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 76.33it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for o


=== FINAL RESULT ===
{
  "metadata": {
    "image_path": "data/temp/X00016469612.jpg",
    "doc_category": "receipt",
    "fields": [
      "company",
      "date",
      "address",
      "total",
      "phone number"
    ],
    "debug": true,
    "timestamp": "2026-03-19T00:50:52",
    "elapsed_seconds": 185.6
  },
  "field_results": {
    "company": {
      "recommended_value": "TA -K (TAMAN DAYA) SDN BHD",
      "selected_engine": "engineA",
      "selection_reason": "",
      "final_rule_consistency": "high",
      "final_engine_self_consistency": "strong",
      "final_ocr_alignment": "strong",
      "final_ocr_corruption": "absent",
      "field_confidence": "very_high",
      "field_state": "pass",
      "state_reason": ""
    },
    "date": {
      "recommended_value": "25/12/2018 8:13:39 PM",
      "selected_engine": "engineB",
      "selection_reason": "",
      "final_rule_consistency": "high",
      "final_engine_self_consistency": "strong",
      "final_ocr_alignment": "s

# Test

In [7]:
from pathlib import Path
import json

In [8]:
raw_text_path = "data/temp/debug/X00016469612_raw_text.txt"
constraints_path = "data/temp/debug/X00016469612_engineA_constraints.txt"
ENGINE_A_PATH = "/projects/insightx-lab/cn_grpo/models/Llama-3.1-8B-Instruct"

In [9]:
raw_text = Path(raw_text_path).read_text(encoding="utf-8")
constraint_trace = Path(constraints_path).read_text(encoding="utf-8")

In [58]:
prompt = build_extraction_prompt(raw_text, constraint_trace, FIELDS, DOC_CATEGORY)

print(prompt)

tokenizer, model = load_model_and_tokenizer(ENGINE_A_PATH)

response = run_chat_inference(
    model,
    tokenizer,
    prompt,
    max_new_tokens=2400
)

You are an information extraction engine for receipt documents.

You are given:
1. OCR raw text from a receipt
2. A constraint trace previously generated for this document

Your task:
Extract the following fields:
- company
- date
- address
- total
- phone number

For each field, provide:
1. field_extraction: the extracted value
2. evidence_trace: the most directly relevant supporting OCR text
3. reasoning: explain why this evidence supports the extraction and how the constraint trace helped

Important guidelines:
- Extract each field independently.
- Use only the OCR raw text and the constraint trace below.
- Do NOT use outside knowledge.
- Every field listed in the schema MUST appear in the output, even if no value is found.
- If a field is partially visible or uncertain but still plausibly present, provide the best supported guess and mention the uncertainty in reasoning.
- If a field is not present in the OCR text, or no sufficiently supported candidate can be identified, you MUST 

Loading weights: 100%|██████████| 291/291 [00:03<00:00, 73.34it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [59]:
print(response)

{
  "field_extraction": {
    "company": "TA -K (TAMAN DAYA) SDN BHD",
    "date": "25/12/2018 8:13:39 PM",
    "address": "NO.5? $5,57 & 59, JALAN SAGU 18, TAMAN DAYA 81100 JOHOR BAHRU.",
    "phone number": ""
  },
  "evidence_trace": {
    "company": "BOOK TA -K (TAMAN DAYA) SDN BHD",
    "date": "Date 25/12/2018 8:13:39 PM",
    "address": "NO.5? $5,57 & 59, JALAN SAGU 18, TAMAN DAYA 81100 JOHOR BAHRU.",
    "phone number": ""
  },
  "reasoning": {
    "company": "The company name appears as a multi-line text block with words like 'TA', 'SDN', or 'BHD' indicating a company name, which matches the constraint trace.",
    "date": "The date appears in the format 'DD/MM/YYYY HH:MM:SS' near the top of the document, which matches the constraint trace.",
    "address": "The address includes a street number, street name, and postal code, and includes words like 'JALAN' indicating a location, which matches the constraint trace.",
    "phone number": "No sufficiently supported value was foun